<a href="https://colab.research.google.com/github/Oruntu-Tanima-Proje/otProje/blob/main/notebooks/02_veriKesfi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 02 - Veri Keşfi

## Domates Yaprak Veri Setinin Analizi

Bu notebook, hazırlanan domates yaprağı veri setini görsel olarak inceler ve
sınıf dağılımını, örnek görüntüleri ve görüntü özelliklerini analiz eder.

### İçerik
- ✅ Sınıf dağılımı grafiği
- ✅ Her sınıftan örnek görüntüler
- ✅ Görüntü boyut analizi
- ✅ Veri seti istatistikleri

### Ön Koşul
- `01_veri_hazirlik.ipynb` çalıştırılmış olmalı
- `tomato_data/` klasörü hazır olmalı (veya Drive'da yedek)


In [ ]:
# ============================================================
# 1. HAZIRLIK - Drive'dan veri kopyala (gerekirse)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

drive_proje = "/content/drive/MyDrive/Domates_Projesi"

# Veriyi Drive'dan kopyala (eğer yoksa)
if not os.path.exists("tomato_data"):
    print("📦 Veri seti Drive'dan kopyalanıyor...")
    shutil.copytree(f"{drive_proje}/data", "tomato_data")
    print("   ✅ Tamamlandı")
else:
    print("✅ Veri seti yerinde")

# Yollar
train_dir = "tomato_data/train"
valid_dir = "tomato_data/valid"
test_dir = "tomato_data/test"

print("\n✅ Hazırlık tamamlandı")

In [ ]:
# ============================================================
# 2. SINIF DAĞILIMI GRAFİĞİ
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Türkçe sınıf isimleri
class_names_tr = {
    "Tomato___Bacterial_spot": "Bakteriyel Leke",
    "Tomato___Early_blight": "Erken Yaprak Yanıklığı",
    "Tomato___Late_blight": "Geç Yaprak Yanıklığı",
    "Tomato___Leaf_Mold": "Yaprak Küfü",
    "Tomato___Septoria_leaf_spot": "Septorya Yaprak Lekesi",
    "Tomato___Spider_mites Two-spotted_spider_mite": "Kırmızı Örümcek Hasarı",
    "Tomato___Target_Spot": "Hedef Leke",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "Sarı Yaprak Kıvırcıklığı Virüsü",
    "Tomato___Tomato_mosaic_virus": "Mozaik Virüsü",
    "Tomato___healthy": "Sağlıklı"
}

# Her sınıf için sayıları topla
classes = sorted(os.listdir(train_dir))
train_counts, valid_counts, test_counts = [], [], []

for cls in classes:
    train_counts.append(len(os.listdir(os.path.join(train_dir, cls))))
    valid_counts.append(len(os.listdir(os.path.join(valid_dir, cls))))
    test_counts.append(len(os.listdir(os.path.join(test_dir, cls))))

labels_tr = [class_names_tr[c] for c in classes]

# Grafik
fig, ax = plt.subplots(figsize=(13, 8))
y_pos = np.arange(len(classes))
height = 0.27

ax.barh(y_pos - height, train_counts, height, label='Train (Eğitim)', color='#2E86AB')
ax.barh(y_pos,          valid_counts, height, label='Valid (Doğrulama)', color='#F18F01')
ax.barh(y_pos + height, test_counts,  height, label='Test', color='#A23B72')

ax.set_yticks(y_pos)
ax.set_yticklabels(labels_tr, fontsize=10)
ax.set_xlabel('Görüntü Sayısı', fontsize=11)
ax.set_title('Domates Yaprağı Veri Seti - Sınıf Dağılımı', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)

# Sayıları bar uçlarına yaz
for i, (t, v, te) in enumerate(zip(train_counts, valid_counts, test_counts)):
    ax.text(t + 30, i - height, str(t), va='center', fontsize=8)
    ax.text(v + 30, i,          str(v), va='center', fontsize=8)
    ax.text(te + 30, i + height, str(te), va='center', fontsize=8)

plt.tight_layout()
plt.savefig('sinif_dagilimi.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Sınıf dağılımı grafiği oluşturuldu")

# Özet
total = sum(train_counts) + sum(valid_counts) + sum(test_counts)
print(f"\n📊 ÖZET:")
print(f"   Train: {sum(train_counts)} görüntü")
print(f"   Valid: {sum(valid_counts)} görüntü")
print(f"   Test:  {sum(test_counts)} görüntü")
print(f"   TOPLAM: {total} görüntü, {len(classes)} sınıf")

In [ ]:
# ============================================================
# 3. HER SINIFTAN ÖRNEK GÖRÜNTÜLER
# ============================================================

import random
from PIL import Image

random.seed(42)

# 10 sınıfın her birinden 1 örnek
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, cls in enumerate(classes):
    class_path = os.path.join(train_dir, cls)
    sample_image = random.choice(os.listdir(class_path))
    img_path = os.path.join(class_path, sample_image)
    img = Image.open(img_path)

    axes[i].imshow(img)
    axes[i].set_title(class_names_tr[cls], fontsize=11, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Domates Yaprağı Hastalıkları - Örnek Görüntüler',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ornek_goruntuler.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Örnek görüntüler kaydedildi")

In [ ]:
# ============================================================
# 4. GÖRÜNTÜ BOYUT KONTROLÜ
# ============================================================

from collections import Counter

# Rastgele 200 görüntünün boyutlarını kontrol et
sizes = []
for _ in range(200):
    rand_class = random.choice(classes)
    class_path = os.path.join(train_dir, rand_class)
    rand_img = random.choice(os.listdir(class_path))
    img = Image.open(os.path.join(class_path, rand_img))
    sizes.append(img.size)

size_counts = Counter(sizes)

print("📏 GÖRÜNTÜ BOYUT ANALİZİ (200 örneklem)\n")
print(f"{'Boyut':<15} {'Sayı':>6}  {'Yüzde':>7}")
print("-" * 35)
for size, count in size_counts.most_common():
    pct = count / 200 * 100
    print(f"{str(size):<15} {count:>6}  {pct:>6.1f}%")

print(f"\n✅ Benzersiz boyut sayısı: {len(size_counts)}")
if len(size_counts) == 1:
    print("   ✓ Tüm görüntüler AYNI boyutta - uniformluk var!")
    print("   ✓ Modelleme için ideal")

In [ ]:
# ============================================================
# 5. VERİ SETİ İSTATİSTİKLERİ
# ============================================================

import pandas as pd

# Detaylı tablo
data = []
for cls in classes:
    train_n = len(os.listdir(os.path.join(train_dir, cls)))
    valid_n = len(os.listdir(os.path.join(valid_dir, cls)))
    test_n = len(os.listdir(os.path.join(test_dir, cls)))
    total = train_n + valid_n + test_n
    data.append({
        'Sınıf': class_names_tr[cls],
        'Train': train_n,
        'Valid': valid_n,
        'Test': test_n,
        'Toplam': total
    })

df = pd.DataFrame(data)

# Sınıf dengesizliği analizi
max_class = df['Toplam'].max()
min_class = df['Toplam'].min()
imbalance_ratio = max_class / min_class

print("=" * 70)
print("📊 SINIF BAZLI VERİ DAĞILIMI")
print("=" * 70)
print(df.to_string(index=False))
print("=" * 70)

print(f"\n📈 İSTATİSTİKLER:")
print(f"   En kalabalık sınıf: {max_class} görüntü")
print(f"   En az kalabalık sınıf: {min_class} görüntü")
print(f"   Dengesizlik oranı: {imbalance_ratio:.2f}")

if imbalance_ratio < 1.5:
    print(f"   ✅ Sınıflar DENGELİ (oran < 1.5)")
    print(f"   ✓ Class weights kullanmaya gerek yok")
else:
    print(f"   ⚠️  Sınıflar dengesiz olabilir")

## ✅ Veri Keşfi Tamamlandı

### Bulgular
- **Toplam Görüntü**: ~22.930 (10 sınıf üzerinde)
- **Görüntü Boyutu**: 256x256 RGB (uniform)
- **Sınıf Dengesi**: Dengeli (oran ~1.15)
- **Sınıf Sayısı**: 10 (9 hastalık + sağlıklı)

### Sonuç
Veri seti modelleme için **hazır ve dengeli**. Class weights gerekmez.

### Sıradaki Adım
👉 `03_mobilenetv2_egitim.ipynb` notebook'unu açın ve ilk model eğitimini başlatın.